<a href="https://colab.research.google.com/github/JVytas/Bakalaurinis_Darbas/blob/main/GA_algorithm_Sudoku.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import random
from copy import deepcopy
import pandas as pd
from tqdm import tqdm
from IPython.display import display

# Sudoku lenta (9x9)
initial_board_9x9 = np.array([
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
])

# Sudoku lenta (4x4)
initial_board_4x4 = np.array([
    [1, 2, 0, 0],
    [3, 4, 0, 0],
    [0, 0, 1, 2],
    [0, 0, 3, 4]
])

# Genetinio algoritmo komponentai
def get_fixed_positions(board):
    """
    Identifies the positions of pre-filled (non-zero) numbers on the Sudoku board.
    These positions represent the unchangeable cells of the puzzle.

    Args:
        board (np.array): The Sudoku board (N x N numpy array).

    Returns:
        list of lists: A list where each inner list contains the column indices
                        of the fixed numbers for each row.
    """
    N = board.shape[0]
    return [[i for i in range(N) if row[i] != 0] for row in board]

def create_individual(board, fixed_pos):
    """
    Generates a single 'individual' (a candidate Sudoku solution).
    It fills the empty cells in each row with the missing numbers (1 to N)
    while respecting the fixed positions from the original puzzle.

    Args:
        board (np.array): The initial Sudoku board (N x N numpy array).
        fixed_pos (list of lists): Positions of the fixed (pre-filled) numbers.

    Returns:
        np.array: A new Sudoku board representing an individual, with rows
                  initially valid (containing numbers 1 to N without repetition,
                  respecting fixed positions).
    """
    N = board.shape[0]
    individual = deepcopy(board)
    for i in range(N):
        # Identify numbers already present in the row from the fixed positions
        missing = [n for n in range(1, N + 1) if n not in board[i]]
        random.shuffle(missing) # Randomize the order of missing numbers
        idx = 0
        for j in range(N):
            # Fill only the non-fixed positions with shuffled missing numbers
            if j not in fixed_pos[i]:
                individual[i][j] = missing[idx]
                idx += 1
    return individual

def fitness(individual):
    """
    Evaluates the 'fitness' of an individual (a potential Sudoku solution).
    A lower fitness score indicates a better solution, with 0 being a perfect solution.
    It calculates penalties for duplicate numbers in columns and 3x3 blocks.
    Rows are implicitly valid due to the 'create_individual' function.

    Args:
        individual (np.array): A candidate Sudoku solution (N x N numpy array).

    Returns:
        int: The fitness score, where 0 means a perfect solution.
    """
    N = individual.shape[0]
    block_size = int(np.sqrt(N))
    score = 0

    # Calculate score based on column validity
    for col in range(N):
        # Count missing unique numbers in each column (penalizing duplicates)
        score += (N - len(set(individual[:, col])))

    # Calculate score based on 3x3 (or N/sqrt(N) x N/sqrt(N)) block validity
    for i in range(block_size):
        for j in range(block_size):
            # Extract the current block and flatten it to a 1D array
            square = individual[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size].flatten()
            # Count missing unique numbers in the block (penalizing duplicates)
            score += (N - len(set(square)))
    return score

def mutate(individual, fixed_pos):
    """
    Introduces genetic variation by performing a mutation on an individual.
    It selects a random row and swaps two mutable (non-fixed) numbers within that row.

    Args:
        individual (np.array): The individual to mutate.
        fixed_pos (list of lists): Positions of the fixed numbers, which cannot be mutated.

    Returns:
        np.array: The mutated individual.
    """
    N = individual.shape[0]
    i = random.randint(0, N-1) # Select a random row
    row = individual[i]
    # Find indices of cells in the selected row that are not fixed
    mutable_indices = [j for j in range(N) if j not in fixed_pos[i]]

    # If there are at least two mutable cells, swap two of them
    if len(mutable_indices) >= 2:
        a, b = random.sample(mutable_indices, 2)
        row[a], row[b] = row[b], row[a]
    return individual

def crossover(parent1, parent2):
    """
    Combines genetic material from two parent individuals to create a child.
    For each row, the child inherits the entire row from either parent1 or parent2
    with a 50% probability.

    Args:
        parent1 (np.array): The first parent individual.
        parent2 (np.array): The second parent individual.

    Returns:
        np.array: The child individual generated from the two parents.
    """
    N = parent1.shape[0]
    child = np.zeros((N, N), dtype=int)
    for i in range(N):
        # Randomly choose to inherit a row from parent1 or parent2
        child[i] = parent1[i] if random.random() < 0.5 else parent2[i]
    return child

def genetic_algorithm(board, generations=1000, pop_size=1000, mutation_rate=0.1):
    """
    Implements the main genetic algorithm to solve a Sudoku puzzle.

    Args:
        board (np.array): The initial Sudoku board to solve.
        generations (int): The maximum number of generations to run the algorithm.
        pop_size (int): The size of the population (number of individuals).
        mutation_rate (float): The probability of an individual undergoing mutation.

    Returns:
        tuple: A tuple containing:
            - generation (int): The generation number when a solution was found
                                or the total generations if no solution was found.
            - success (bool): True if a solution was found, False otherwise.
            - solution (np.array or None): The solved Sudoku board if successful,
                                          otherwise None.
    """
    fixed_pos = get_fixed_positions(board) # Get fixed positions once
    # Initialize population with individuals respecting fixed positions and row validity
    population = [create_individual(board, fixed_pos) for _ in range(pop_size)]
    N = board.shape[0]

    for generation in range(generations):
        # Sort the population by fitness score (lower is better)
        population = sorted(population, key=fitness)

        # Check if a perfect solution (fitness score of 0) has been found
        if fitness(population[0]) == 0:
            return generation, True, population[0]

        # Elitism: Keep a small percentage of the fittest individuals for the next generation
        new_population = population[:int(pop_size*0.02)] # Keep top 2%

        # Fill the rest of the new population through crossover and mutation
        while len(new_population) < pop_size:
            # Select two parents from a larger pool of fitter individuals to encourage diversity
            # Ensure the sample pool has at least 2 individuals for random.sample
            sample_pool_size = max(2, int(pop_size * 0.1)) # Select from top 10% (or minimum 2)
            p1, p2 = random.sample(population[:sample_pool_size], 2)

            child = crossover(p1, p2) # Create a child through crossover

            # Apply mutation to the child based on mutation rate
            if random.random() < mutation_rate:
                child = mutate(child, fixed_pos)

            new_population.append(child)

        population = new_population # Update the population for the next generation

    # If no solution is found within the given generations
    return generations, False, None

# Eksperimentas
board_configs = {
    "9x9": initial_board_9x9,
    "4x4": initial_board_4x4 # Added 4x4 board to configurations

}

# Updated pop_sizes for different board types
pop_sizes_by_board = {
    "9x9": [100, 200, 300],
    "4x4": [10, 20, 30]
}

mutation_rates = [0.05, 0.1]
results = []

num_trials = 10 # Number of iterations for each combination

for board_size_str, initial_board in board_configs.items():
    print(f"\nRunning experiments for {board_size_str} board...")
    current_pop_sizes = pop_sizes_by_board[board_size_str] # Get pop_sizes specific to the board
    for pop in current_pop_sizes:
        for mut in mutation_rates:
            successes = 0
            total_generations = 0
            for _ in tqdm(range(num_trials), desc=f"P={pop}, M={mut} ({board_size_str})"):
                gens, success, solution = genetic_algorithm(
                    initial_board,
                    generations=1000,
                    pop_size=pop,
                    mutation_rate=mut
                )
                if success:
                    successes += 1
                    total_generations += gens
                else:
                    total_generations += 1000 # If not successful, assume it took all generations
            avg_gen = round(total_generations / num_trials, 1) if successes > 0 else 1000 # Handle cases with 0 successes
            results.append({
                "Lentelės dydis": board_size_str,
                "Populiacija": pop,
                "Mutacija": mut,
                "Sėkmės dažnis": f"{successes}/{num_trials}",
                "Vid. kartų": avg_gen,
                "Iteracijos": num_trials
            })

# Konvertuojame į lentelę
df = pd.DataFrame(results)
display(df)

# Išsaugome CSV failą
df.to_csv("sudoku_rezultatai.csv", index=False)


Running experiments for 9x9 board...


P=300, M=0.1 (9x9): 100%|██████████| 10/10 [03:32<00:00, 21.27s/it]



Running experiments for 4x4 board...


P=30, M=0.1 (4x4): 100%|██████████| 10/10 [00:00<00:00, 299.20it/s]


,Lentelės dydis,Populiacija,Mutacija,Sėkmės dažnis,Vid. kartų,Iteracijos
0,9x9,100,0.05,0/10,1000.0,10
1,9x9,100,0.10,0/10,1000.0,10
2,9x9,200,0.05,0/10,1000.0,10
3,9x9,200,0.10,0/10,1000.0,10
4,9x9,300,0.05,0/10,1000.0,10
5,9x9,300,0.10,0/10,1000.0,10
6,4x4,10,0.05,10/10,1.4,10
7,4x4,10,0.10,10/10,0.2,10
8,4x4,20,0.05,10/10,0.4,10
9,4x4,20,0.10,10/10,0.0,10


### Sudoku Genetic Algorithm Implementation

This code implements a genetic algorithm to solve a 9x9 Sudoku puzzle. It defines the core components of a genetic algorithm: individual representation, fitness function, mutation, crossover, and the main algorithm loop.

### Initial Sudoku Board

This section initializes a 9x9 Sudoku board, where `0` represents an empty cell. This `initial_board_9x9` serves as the problem to be solved by the genetic algorithm.

### Genetic Algorithm Components

#### `get_fixed_positions(board)`

This function identifies and stores the positions of the pre-filled (non-zero) numbers in the initial Sudoku board. These positions will remain constant throughout the genetic algorithm process, as they are part of the original puzzle constraints.

#### `create_individual(board, fixed_pos)`

This function generates an initial 'individual' (a potential solution) for the genetic algorithm. It fills the empty cells (`0`) in each row of the board with the missing numbers (1-9) for that specific row, ensuring that each row initially contains all numbers from 1 to 9 without repetition. The fixed positions are preserved.

#### `fitness(individual)`

This is the fitness function, which evaluates how 'good' a given Sudoku solution (individual) is. A lower fitness score is better, with a score of `0` indicating a perfect solution.

It calculates the score based on two criteria:
1.  **Column completeness**: It penalizes for duplicate numbers in each column.
2.  **3x3 square completeness**: It penalizes for duplicate numbers within each of the nine 3x3 sub-grids.

The number of missing unique values in columns and 3x3 squares directly contributes to the score. Rows are already valid due to the `create_individual` function.

#### `mutate(individual, fixed_pos)`

This function introduces genetic variation into an individual. It randomly selects a row and then randomly swaps two mutable (non-fixed) numbers within that row. This helps the algorithm explore different solution spaces and avoid local optima.

#### `crossover(parent1, parent2)`

This function combines genetic material from two 'parent' individuals to create a new 'child' individual. For each row, it randomly selects whether to inherit that row from `parent1` or `parent2`. This mechanism helps propagate good traits from successful parents to new generations.

#### `genetic_algorithm(board, generations, pop_size, mutation_rate)`

This is the main genetic algorithm loop. It performs the following steps:
1.  **Initialization**: Creates an initial population of `pop_size` individuals.
2.  **Evolution Loop**: Iterates for a specified number of `generations`.
    *   **Evaluation**: Sorts the population based on their fitness scores.
    *   **Termination Condition**: If a perfect solution (fitness score of 0) is found, the algorithm stops and returns the generation number and success status.
    *   **Selection & Reproduction**: The top 20 individuals are carried over to the next generation directly (`new_population = population[:20]`). New individuals are created by selecting two parents from the top 100 individuals, performing `crossover`, and potentially applying `mutate` based on the `mutation_rate`.
    *   **Population Update**: The `new_population` replaces the old one.
3.  **Result**: Returns the number of generations processed and whether a solution was found.

### Experiment Setup

This section defines the parameters for an experiment to evaluate the performance of the genetic algorithm under different conditions:

*   `pop_sizes`: Various population sizes (200, 500, 1000) to test.
*   `mutation_rates`: Different mutation probabilities (0.05, 0.1) to test.

The experiment runs 10 trials for each combination of `pop_size` and `mutation_rate`, collecting data on the success rate and average number of generations required to find a solution.

### Results Processing and Output

After running all experiments, this section compiles the collected data into a pandas DataFrame. The DataFrame includes columns for 'Lentelės dydis' (Board Size), 'Populiacija' (Population Size), 'Mutacija' (Mutation Rate), 'Sėkmės dažnis' (Success Rate), and 'Vid. kartų' (Average Generations).

The results DataFrame is then displayed using `display(df)` and saved to a CSV file named `sudoku_9x9_rezultatai.csv` for further analysis.